In [1]:
# -*- coding: utf-8 -*-
"""
EDA для датасета банковских продуктов (устойчивый к ' NA' с пробелами)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from matplotlib.backends.backend_pdf import PdfPages
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*70)
print("ЗАГРУЗКА ДАННЫХ (СЕМПЛ 500 000 СТРОК)")
print("="*70)

# 1. Загрузим только заголовки, чтобы определить целевые колонки
sample_headers = pd.read_csv('data/train_ver2.csv', nrows=0)
target_cols = [col for col in sample_headers.columns if col.startswith('ind_') and col.endswith('_ult1')]
print(f"Найдено целевых колонок (продуктов): {len(target_cols)}")

# 2. Загрузим семпл данных (первые 500k строк, этого достаточно для EDA)
#    Если нужно больше, увеличьте nrows, но учтите память
nrows = 500_000  # можно поставить 1_000_000, если RAM > 8GB

# Расширенный список значений NA (включая пробельные варианты)
na_values = [' NA', '     NA', 'NA', 'n/a', 'N/A', 'NULL', 'null', '', ' ', '?', 'unknown']

# Читаем без указания dtype, пусть pandas сам определит типы
df = pd.read_csv('data/train_ver2.csv',
                 nrows=nrows,
                 na_values=na_values,
                 skipinitialspace=True,   # обрезает пробелы в начале и конце
                 low_memory=False)

print(f"Загружено {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(f"Используемая память: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# 3. Приводим целевые колонки к int8 (заполняем NaN нулями)
for col in target_cols:
    # Если колонка ещё не числовая, преобразуем
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int8')

# 4. Преобразуем даты
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'], errors='coerce')
if 'fecha_alta' in df.columns:
    df['fecha_alta'] = pd.to_datetime(df['fecha_alta'], errors='coerce')

# 5. Категориальные колонки -> category
cat_cols = ['ind_empleado', 'pais_residencia', 'sexo', 'indrel_1mes', 'tiprel_1mes',
            'indresi', 'indext', 'conyuemp', 'canal_entrada', 'indfall', 'nomprov', 'segmento']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 6. Числовые колонки приводим к float32 (если есть некорректные значения -> NaN)
num_cols = ['age', 'antiguedad', 'renta', 'ind_nuevo', 'indrel', 'tipodom', 'cod_prov', 'ind_actividad_cliente']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

print("\n" + "="*70)
print("ИНФОРМАЦИЯ О ЗАГРУЖЕННЫХ ДАННЫХ")
print("="*70)
df.info()
print(f"\nПропуски в ключевых колонках:\n{df[['age','renta','antiguedad']].isnull().sum()}")

# ================================
# ДАЛЬНЕЙШИЙ EDA (как в предыдущем коде)
# ================================

# Количество продуктов на клиента
df['num_products'] = df[target_cols].sum(axis=1)
print(f"\nСреднее количество продуктов: {df['num_products'].mean():.2f}")
print(f"Медиана: {df['num_products'].median()}")
print(f"Максимум: {df['num_products'].max()}")

# Топ-15 продуктов
product_freq = df[target_cols].sum().sort_values(ascending=False)
top_products = product_freq.head(15)

# График распределения количества продуктов
fig1, ax1 = plt.subplots()
sns.histplot(df['num_products'], bins=range(0, df['num_products'].max()+2), discrete=True, ax=ax1)
ax1.set_title('Распределение количества продуктов на клиента')

# График топ-15 продуктов
fig2, ax2 = plt.subplots(figsize=(12,6))
top_products.plot(kind='bar', color='skyblue', ax=ax2)
ax2.set_title('Топ-15 наиболее распространённых продуктов')
ax2.tick_params(axis='x', rotation=45)

# Корреляция Жаккара для топ-20 продуктов (упрощённо)
top20 = product_freq.head(20).index
jaccard_mat = np.zeros((len(top20), len(top20)))
for i, p1 in enumerate(top20):
    for j, p2 in enumerate(top20):
        jaccard_mat[i, j] = (df[p1] & df[p2]).sum() / (df[p1] | df[p2]).sum()  # ускоренная версия
fig3, ax3 = plt.subplots(figsize=(12,10))
sns.heatmap(jaccard_mat, xticklabels=top20, yticklabels=top20, cmap='Blues', annot=False, ax=ax3)
ax3.set_title('Сходство Жаккара (топ-20 продуктов)')

# Временная динамика (если есть даты)
if 'fecha_dato' in df.columns and df['fecha_dato'].notna().any():
    monthly = df.groupby(df['fecha_dato'].dt.to_period('M')).agg({
        'num_products': 'mean',
        **{p: 'mean' for p in top_products.head(5).index}
    }).reset_index()
    monthly['fecha_dato'] = monthly['fecha_dato'].astype(str)
    
    fig4, ax4 = plt.subplots(figsize=(14,6))
    for prod in top_products.head(5).index:
        ax4.plot(monthly['fecha_dato'], monthly[prod], marker='o', label=prod)
    ax4.set_title('Динамика проникновения топ-5 продуктов')
    ax4.legend()
    plt.xticks(rotation=45)

# Связь возраста, дохода, стажа с продуктами
top3 = top_products.head(3).index
for prod in top3:
    fig, axes = plt.subplots(1, 3, figsize=(15,4))
    # Возраст
    axes[0].boxplot([df[df[prod]==1]['age'].dropna(), df[df[prod]==0]['age'].dropna()],
                    labels=['Есть продукт', 'Нет продукта'])
    axes[0].set_title(f'Возраст vs {prod}')
    # Доход
    axes[1].boxplot([df[df[prod]==1]['renta'].dropna(), df[df[prod]==0]['renta'].dropna()],
                    labels=['Есть', 'Нет'])
    axes[1].set_title(f'Доход vs {prod}')
    # Стаж
    axes[2].boxplot([df[df[prod]==1]['antiguedad'].dropna(), df[df[prod]==0]['antiguedad'].dropna()],
                    labels=['Есть', 'Нет'])
    axes[2].set_title(f'Стаж vs {prod}')
    plt.tight_layout()

# Категориальные признаки
categorical_cols = ['sexo', 'segmento', 'ind_empleado']
for col in categorical_cols:
    if col in df.columns:
        fig, axes = plt.subplots(1, len(top3), figsize=(15,4))
        for i, prod in enumerate(top3):
            grouped = df.groupby(col, observed=True)[prod].mean().sort_values(ascending=False)
            axes[i].barh(grouped.index.astype(str), grouped.values, color='teal')
            axes[i].set_title(f'Доля {prod} по {col}')
        plt.tight_layout()

# Корреляционная матрица
sample_corr = df[['age', 'renta', 'antiguedad', 'num_products'] + list(top3)].dropna().sample(min(30000, len(df)))
corr_matrix = sample_corr.corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Корреляция числовых признаков и топ-продуктов')

# Сохраняем всё в PDF
with PdfPages('eda_bank_products_report.pdf') as pdf:
    for i in plt.get_fignums():
        fig = plt.figure(i)
        pdf.savefig(fig)
        plt.close(fig)

print("\n" + "="*70)
print("ОТЧЁТ СОХРАНЁН: eda_bank_products_report.pdf")
print("="*70)
print(f"Уникальных клиентов: {df['ncodpers'].nunique():,}")
print(f"Самый популярный продукт: {product_freq.index[0]} ({product_freq.iloc[0]:,} записей)")
print("Рекомендации:")
print("- Временной срез: для предсказаний использовать лаговые признаки")
print("- Заполнять пропуски renta медианой по prov или segmento")
print("- Учитывать дисбаланс классов при обучении")

ЗАГРУЗКА ДАННЫХ (СЕМПЛ 500 000 СТРОК)
Найдено целевых колонок (продуктов): 24
Загружено 500,000 строк, 48 колонок
Используемая память: 511.5 MB

ИНФОРМАЦИЯ О ЗАГРУЖЕННЫХ ДАННЫХ
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 48 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   fecha_dato             500000 non-null  datetime64[ns]
 1   ncodpers               500000 non-null  int64         
 2   ind_empleado           494611 non-null  category      
 3   pais_residencia        494611 non-null  category      
 4   sexo                   494610 non-null  category      
 5   age                    494611 non-null  float32       
 6   fecha_alta             494611 non-null  datetime64[ns]
 7   ind_nuevo              494611 non-null  float32       
 8   antiguedad             494611 non-null  float32       
 9   indrel                 494611 non-null  float32

In [ ]:
# %% [markdown]
# # Моделирование для задачи рекомендации банковских продуктов (исправленная версия)

# %% [code]
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
from catboost import CatBoostClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
import joblib
import gc

# %% [markdown]
# ## 1. Загрузка данных (только 2016 год)

# %% [code]
dtypes = {
    'fecha_dato': 'object',
    'ncodpers': 'int32',
    'age': 'float32',
    'antiguedad': 'float32',
    'renta': 'float32',
    'sexo': 'object',
    'segmento': 'object',
    'ind_empleado': 'object',
    'canal_entrada': 'object',
    'ind_nuevo': 'float32',
    'indrel': 'float32',
}

sample_headers = pd.read_csv('data/train_ver2.csv', nrows=0)
target_cols = [col for col in sample_headers.columns if col.startswith('ind_') and col.endswith('_ult1')]

usecols = ['fecha_dato', 'ncodpers', 'age', 'antiguedad', 'renta', 'sexo', 
           'segmento', 'ind_empleado', 'canal_entrada', 'ind_nuevo', 'indrel'] + target_cols

df = pd.read_csv('data/train_ver2.csv', 
                 usecols=usecols,
                 dtype=dtypes,
                 na_values=[' NA', '     NA', 'NA', 'null', ''],
                 skipinitialspace=True,
                 low_memory=False)

df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])
df = df[df['fecha_dato'] >= '2016-01-01']
print(f"Загружено {df.shape[0]:,} строк за 2016 год, память: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

# %% [markdown]
# ## 2. Создание таргета (новые продукты в следующем месяце)

# %% [code]
df = df.sort_values(['ncodpers', 'fecha_dato']).reset_index(drop=True)

def add_target_columns(df, target_cols, shift_months=1):
    shifted = df.groupby('ncodpers')[target_cols].shift(-shift_months)
    for col in target_cols:
        new_col = f'{col}_new'
        df[new_col] = ((shifted[col] == 1) & (df[col] == 0)).fillna(0).astype('int8')
    return df

df = add_target_columns(df, target_cols, shift_months=1)
new_target_cols = [f'{col}_new' for col in target_cols]
df = df.dropna(subset=new_target_cols, how='all')
print(f"Осталось {df.shape[0]:,} строк")

# %% [markdown]
# ## 3. Признаки и предобработка

# %% [code]
feature_cols = ['age', 'antiguedad', 'renta', 'sexo', 'segmento', 
                'ind_empleado', 'canal_entrada', 'ind_nuevo', 'indrel'] + target_cols

df['renta'] = df.groupby('segmento')['renta'].transform(lambda x: x.fillna(x.median()))
df['renta'] = df['renta'].fillna(df['renta'].median())
df['age'] = df['age'].fillna(df['age'].median())
df['antiguedad'] = df['antiguedad'].fillna(0)
df['ind_nuevo'] = df['ind_nuevo'].fillna(0)
df['indrel'] = df['indrel'].fillna(0)

cat_features = ['sexo', 'segmento', 'ind_empleado', 'canal_entrada']
for col in cat_features:
    df[col] = df[col].astype(str).fillna('unknown')

# %% [markdown]
# ## 4. Time-Based Split

# %% [code]
train_mask = (df['fecha_dato'] >= '2016-01-01') & (df['fecha_dato'] <= '2016-03-28')
val_mask = (df['fecha_dato'] >= '2016-04-01') & (df['fecha_dato'] < '2016-05-01')
test_mask = (df['fecha_dato'] >= '2016-05-01')

X_train = df[train_mask][feature_cols]
y_train = df[train_mask][new_target_cols]
X_val = df[val_mask][feature_cols]
y_val = df[val_mask][new_target_cols]
X_test = df[test_mask][feature_cols]
y_test = df[test_mask][new_target_cols]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

del df
gc.collect()

# %% [markdown]
# ## 5. Обучение CatBoost OneVsRest

# %% [code]
base_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=6,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=20,
    cat_features=cat_features,
    loss_function='Logloss'
)

model = OneVsRestClassifier(base_model)

mlflow.set_experiment("bank_products_rec")
with mlflow.start_run(run_name="CatBoost_OneVsRest_Fixed"):
    mlflow.log_params({
        "iterations": 100, "learning_rate": 0.05, "depth": 6,
        "auto_class_weights": "Balanced", "train_size": X_train.shape[0]
    })
    
    print("Обучение модели...")
    model.fit(X_train, y_train)
    
    # Функции метрик
    def map_at_k(y_true, y_score, k=7):
        ap_per_sample = []
        for i in range(y_true.shape[0]):
            top_k_idx = np.argsort(y_score[i])[::-1][:k]
            rel = y_true[i][top_k_idx]
            cum_prec = np.cumsum(rel) / (np.arange(1, len(rel)+1))
            ap = np.sum(cum_prec * rel) / min(k, max(1, np.sum(y_true[i])))
            ap_per_sample.append(ap)
        return np.mean(ap_per_sample)
    
    def recall_at_k(y_true, y_score, k=7):
        recalls = []
        for i in range(y_true.shape[0]):
            true_pos = y_true[i]
            if true_pos.sum() == 0:
                continue
            top_k_idx = np.argsort(y_score[i])[::-1][:k]
            rel = y_true[i][top_k_idx]
            recalls.append(rel.sum() / true_pos.sum())
        return np.mean(recalls) if recalls else 0.0
    
    # Валидация с исправлением ошибки размерности
    if X_val.shape[0] > 0:
        y_pred_proba_val = model.predict_proba(X_val)
        # Обработка случаев, когда predict_proba возвращает одномерный массив
        proba_matrix_val = []
        for p in y_pred_proba_val:
            if p.ndim == 1:
                # Одномерный массив — это вероятности положительного класса
                proba_matrix_val.append(p)
            else:
                # Двумерный: берём вероятность класса 1 (индекс 1)
                proba_matrix_val.append(p[:, 1])
        proba_matrix_val = np.array(proba_matrix_val).T
        
        map7_val = map_at_k(y_val.values, proba_matrix_val, k=7)
        recall7_val = recall_at_k(y_val.values, proba_matrix_val, k=7)
        
        pr_auc_list, roc_auc_list = [], []
        for i, col in enumerate(new_target_cols):
            if y_val[col].sum() > 0:
                pr_auc_list.append(average_precision_score(y_val[col], proba_matrix_val[:, i]))
                roc_auc_list.append(roc_auc_score(y_val[col], proba_matrix_val[:, i]))
        
        mean_pr_auc = np.mean(pr_auc_list) if pr_auc_list else 0
        mean_roc_auc = np.mean(roc_auc_list) if roc_auc_list else 0
        
        mlflow.log_metrics({
            "map@7_val": map7_val,
            "recall@7_val": recall7_val,
            "mean_pr_auc_val": mean_pr_auc,
            "mean_roc_auc_val": mean_roc_auc
        })
        
        print(f"Validation MAP@7: {map7_val:.4f}")
        print(f"Validation Recall@7: {recall7_val:.4f}")
        print(f"Mean PR-AUC: {mean_pr_auc:.4f}")
        print(f"Mean ROC-AUC: {mean_roc_auc:.4f}")
    
    # Сохраняем модель
    joblib.dump(model, 'model.bin')
    mlflow.log_artifact('model.bin')
    print("Модель сохранена как model.bin")

# %% [markdown]
# ## 6. Оценка на тесте

# %% [code]
if X_test.shape[0] > 0:
    y_pred_proba_test = model.predict_proba(X_test)
    proba_matrix_test = []
    for p in y_pred_proba_test:
        if p.ndim == 1:
            proba_matrix_test.append(p)
        else:
            proba_matrix_test.append(p[:, 1])
    proba_matrix_test = np.array(proba_matrix_test).T
    map7_test = map_at_k(y_test.values, proba_matrix_test, k=7)
    recall7_test = recall_at_k(y_test.values, proba_matrix_test, k=7)
    print(f"Test MAP@7: {map7_test:.4f}")
    print(f"Test Recall@7: {recall7_test:.4f}")

# %% [markdown]
# ## 7. Важность признаков

# %% [code]
if model.estimators_:
    first_estimator = model.estimators_[0]
    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': first_estimator.feature_importances_
    }).sort_values('importance', ascending=False)
    print("\nТоп-10 важных признаков (первый продукт):")
    print(importance.head(10))

Загружено 4,621,976 строк за 2016 год, память: 2.07 GB
Осталось 4,621,976 строк
Train: (2762249, 33), Val: (928274, 33), Test: (931453, 33)
Обучение модели...
0:	learn: 0.5969671	total: 1.18s	remaining: 1m 57s
20:	learn: 0.2936896	total: 24.8s	remaining: 1m 33s
40:	learn: 0.2690616	total: 45.5s	remaining: 1m 5s
60:	learn: 0.2601643	total: 1m 13s	remaining: 47.1s
80:	learn: 0.2559987	total: 1m 42s	remaining: 24s
99:	learn: 0.2534520	total: 2m 8s	remaining: 0us
0:	learn: 0.6424925	total: 1.35s	remaining: 2m 13s
20:	learn: 0.4403740	total: 20.9s	remaining: 1m 18s
40:	learn: 0.3290992	total: 38.7s	remaining: 55.7s
60:	learn: 0.2960326	total: 56.3s	remaining: 36s
80:	learn: 0.2440668	total: 1m 13s	remaining: 17.2s
99:	learn: 0.2228838	total: 1m 30s	remaining: 0us
0:	learn: 0.6478646	total: 1.23s	remaining: 2m 2s
20:	learn: 0.3887489	total: 23.8s	remaining: 1m 29s
40:	learn: 0.3495274	total: 47.1s	remaining: 1m 7s
60:	learn: 0.3362070	total: 1m 7s	remaining: 43s
80:	learn: 0.3283281	total: 1